## All Years Convert PDFS script and filepath

In [ ]:
#GOES THROUGH ALL DATES FOR ALL YEARS, NOT JUST 2023-2024
import pandas as pd
import os
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
import csv
import re
from html import unescape

# regex patterns
incident_pattern = r"Incident #:\s*(\d+)"
date_pattern = r"Date:\s*(\d{4}-\d{2}-\d{2}\s*\d{2}:\d{2}:\d{2})"
type_pattern = r"Type:\s*([\w\s/]+)"
location_pattern = r"Location:\s*(.+?)(?=\n|$)"
arrest_pattern = r"Arrested:"
name_pattern = r"Name:\s*([^:\n]+?)(?=\s*Date of Birth:)"
dob_pattern = r"Date of Birth:\s*(\d{2}/\d{2}/\d{4})"
charges_pattern = r"Charges:\s*((?:.+?(\n|$))*?)(?=\n(?:\w+:|$))"

# helper: check if the text seems valid
def is_meaningful_police_log(text):
    incident_count = len(re.findall(r"Incident\s+#", text))
    has_keywords = "Location:" in text or "Type:" in text or "NOISE ORD" in text
    is_not_all_symbols = bool(re.search(r"[A-Za-z]{3,}", text))
    return incident_count > 0 and has_keywords and is_not_all_symbols

# extract from pdfplumber format
def extract_data_from_text_pdfplumber(text):
    rows = []
    if not re.search(r"[=-]{10,}", text):
        return rows
    incidents = re.split(r"[=-]{10,}", text)
    for entry in incidents:
        rows.append(extract_entry(entry))
    return rows

# extract from OCR fallback format
def extract_data_from_text_ocr(text):
    rows = []
    incidents = re.split(r"(?=Incident\s+#?:\s*\d+)", text)
    for entry in incidents:
        rows.append(extract_entry(entry))
    return rows

# generic extractor from an entry block
def extract_entry(entry):
    return {
        "Incident #": re.search(incident_pattern, entry).group(1) if re.search(incident_pattern, entry) else "",
        "Date": re.search(date_pattern, entry).group(1) if re.search(date_pattern, entry) else "",
        "Type": re.search(type_pattern, entry).group(1) if re.search(type_pattern, entry) else "",
        "Location": re.search(location_pattern, entry).group(1) if re.search(location_pattern, entry) else "",
        "Arrested": "Yes" if re.search(arrest_pattern, entry) else "No",
        "Name": re.search(name_pattern, entry).group(1).strip() if re.search(name_pattern, entry) else "",
        "DOB": re.search(dob_pattern, entry).group(1) if re.search(dob_pattern, entry) else "",
        "Charges": "; ".join(
            line.strip() for line in re.search(charges_pattern, entry, re.DOTALL).group(1).splitlines()
            if line.strip()
        ) if re.search(charges_pattern, entry, re.DOTALL) else ""
    }

# use pdfplumber first, fallback to OCR
def extract_data_from_pdf_auto(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join(p.extract_text() or "" for p in pdf.pages)
            text = unescape(text)

            if is_meaningful_police_log(text):
                return extract_data_from_text_pdfplumber(text)
            else:
                print(f"⚠️ pdfplumber output unreadable, using OCR fallback: {os.path.basename(pdf_path)}")
    except Exception as e:
        print(f"❌ pdfplumber failed on {pdf_path}: {e}")

    # fallback to OCR
    try:
        images = convert_from_path(pdf_path, dpi=300)
        ocr_text = ""
        for img in images:
            ocr_text += pytesseract.image_to_string(img)
        if is_meaningful_police_log(ocr_text):
            return extract_data_from_text_ocr(ocr_text)
    except Exception as e:
        print(f"❌ OCR failed for {pdf_path}: {e}")
    return []

# loop through all PDFs
def parse_all_pdfs_to_csv(input_dir, output_csv, max_pdfs=None):
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Input PDF directory not found: {input_dir}")

    all_rows = []
    pdf_count = 0
    for root, _, files in os.walk(input_dir):
        for file in sorted(files):
            if file.lower().endswith(".pdf"):
                path = os.path.join(root, file)
                print(f"📄 Parsing: {file}")
                rows = extract_data_from_pdf_auto(path)
                if not rows:
                    print(f"⚠️ No extractable data in: {file}")
                all_rows.extend(rows)
                pdf_count += 1
                if max_pdfs and pdf_count >= max_pdfs:
                    print(f"⏸️ Stopping early after {pdf_count} PDFs")
                    break
        if max_pdfs and pdf_count >= max_pdfs:
            break

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["Incident #", "Date", "Type", "Location", "Arrested", "Name", "DOB", "Charges"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"\n✅ CSV created at: {output_csv}")


In [ ]:
#works
import os

notebook_dir = os.getcwd()
PDF_DATA_DIR = os.path.join('/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data')
parsed_csv_path = os.path.join(PDF_DATA_DIR, 'lawrence_2025.csv')

# run for all PDFs, add optional `max_pdfs` argument
parse_all_pdfs_to_csv(PDF_DATA_DIR, parsed_csv_path)
